In [1]:
import pandas as pd
import numpy as np

# Load original Spotify dataset
spotify_df = pd.read_csv("../data/raw/spotify_tracks.csv")

# Keep only the base columns you want for the new YouTube-audio dataset
base_cols = ["track_id", "track_name", "artists", "album_name", "track_genre", "popularity"]
youtube_audio_base = spotify_df[base_cols].copy()

# Add empty mapping columns
youtube_audio_base["audio_filename"] = ""
youtube_audio_base["audio_path"] = ""
youtube_audio_base["audio_source"] = "youtube_audio_local"
youtube_audio_base["audio_loaded"] = False

# Add empty basic audio metadata columns
youtube_audio_base["sample_rate"] = np.nan
youtube_audio_base["duration_sec"] = np.nan

# Add empty rhythm / energy columns
for col in [
    "tempo_bpm",
    "rms_mean", "rms_std",
    "zcr_mean", "zcr_std"
]:
    youtube_audio_base[col] = np.nan

# Add empty spectral columns
for col in [
    "spectral_centroid_mean", "spectral_centroid_std",
    "spectral_bandwidth_mean", "spectral_bandwidth_std",
    "spectral_rolloff_mean", "spectral_rolloff_std",
    "spectral_contrast_mean", "spectral_contrast_std"
]:
    youtube_audio_base[col] = np.nan

# Add empty pitch columns
for col in [
    "pitch_mean", "pitch_std", "pitch_min", "pitch_max"
]:
    youtube_audio_base[col] = np.nan

# Add empty MFCC columns
for i in range(1, 14):
    youtube_audio_base[f"mfcc_{i}_mean"] = np.nan
    youtube_audio_base[f"mfcc_{i}_std"] = np.nan

# Add empty chroma columns
for i in range(1, 13):
    youtube_audio_base[f"chroma_{i}_mean"] = np.nan
    youtube_audio_base[f"chroma_{i}_std"] = np.nan

# Add status columns
youtube_audio_base["feature_extraction_success"] = False
youtube_audio_base["error_message"] = ""

youtube_audio_base.head()

,track_id,track_name,artists,album_name,track_genre,popularity,audio_filename,audio_path,audio_source,audio_loaded,...,chroma_9_mean,chroma_9_std,chroma_10_mean,chroma_10_std,chroma_11_mean,chroma_11_std,chroma_12_mean,chroma_12_std,feature_extraction_success,error_message
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino,Comedy,acoustic,73,,,youtube_audio_local,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward,Ghost (Acoustic),acoustic,55,,,youtube_audio_local,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN,To Begin Again,acoustic,57,,,youtube_audio_local,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,acoustic,71,,,youtube_audio_local,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,
4,5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,Hold On,acoustic,82,,,youtube_audio_local,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,


In [3]:
output_path = "../data/raw/youtube_audio_baseline.csv"
youtube_audio_base.to_csv(output_path, index=False)
print(f"Saved baseline dataset to: {output_path}")

Saved baseline dataset to: ../data/raw/youtube_audio_baseline.csv


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import librosa
from tqdm.auto import tqdm

In [13]:
BASELINE_PATH = "../data/raw/youtube_audio_baseline.csv"
df = pd.read_csv(BASELINE_PATH)
df.head()

C:\Users\Ammar\AppData\Local\Temp\ipykernel_37484\2194627990.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(BASELINE_PATH)


,track_id,track_name,artists,album_name,track_genre,popularity,audio_filename,audio_path,audio_source,audio_loaded,...,chroma_9_mean,chroma_9_std,chroma_10_mean,chroma_10_std,chroma_11_mean,chroma_11_std,chroma_12_mean,chroma_12_std,feature_extraction_success,error_message
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino,Comedy,acoustic,73,Comedy.wav,NaN,youtube_audio_local,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward,Ghost (Acoustic),acoustic,55,Ghost - Acoustic.wav,NaN,youtube_audio_local,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN,To Begin Again,acoustic,57,To Begin Again.wav,NaN,youtube_audio_local,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,acoustic,71,Can't Help Falling In Love.wav,NaN,youtube_audio_local,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
4,5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,Hold On,acoustic,82,Hold On.wav,NaN,youtube_audio_local,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [15]:
AUDIO_DIR = "../data/raw/youtube_audio/"

df["audio_filename"] = df["audio_filename"].fillna("").astype(str).str.strip()
df["audio_path"] = np.where(
    df["audio_filename"] != "",
    AUDIO_DIR + df["audio_filename"],
    ""
)

df[["track_name", "artists", "audio_filename", "audio_path"]].head(10)

,track_name,artists,audio_filename,audio_path
0,Comedy,Gen Hoshino,Comedy.wav,../data/raw/youtube_audio/Comedy.wav
1,Ghost - Acoustic,Ben Woodward,Ghost - Acoustic.wav,../data/raw/youtube_audio/Ghost - Acoustic.wav
2,To Begin Again,Ingrid Michaelson;ZAYN,To Begin Again.wav,../data/raw/youtube_audio/To Begin Again.wav
3,Can't Help Falling In Love,Kina Grannis,Can't Help Falling In Love.wav,../data/raw/youtube_audio/Can't Help Falling I...
4,Hold On,Chord Overstreet,Hold On.wav,../data/raw/youtube_audio/Hold On.wav
5,Days I Will Remember,Tyrone Wells,Days I Will Remember.wav,../data/raw/youtube_audio/Days I Will Remember...
6,Say Something,A Great Big World;Christina Aguilera,Say Something.wav,../data/raw/youtube_audio/Say Something.wav
7,I'm Yours,Jason Mraz,I'm Yours.wav,../data/raw/youtube_audio/I'm Yours.wav
8,Lucky,Jason Mraz;Colbie Caillat,Lucky.wav,../data/raw/youtube_audio/Lucky.wav
9,Hunger,Ross Copperman,Hunger.wav,../data/raw/youtube_audio/Hunger.wav


In [16]:
df["file_exists"] = df["audio_path"].apply(lambda x: Path(x).exists() if x else False)

df[["track_name", "artists", "audio_filename", "audio_path", "file_exists"]].head(10)

,track_name,artists,audio_filename,audio_path,file_exists
0,Comedy,Gen Hoshino,Comedy.wav,../data/raw/youtube_audio/Comedy.wav,True
1,Ghost - Acoustic,Ben Woodward,Ghost - Acoustic.wav,../data/raw/youtube_audio/Ghost - Acoustic.wav,True
2,To Begin Again,Ingrid Michaelson;ZAYN,To Begin Again.wav,../data/raw/youtube_audio/To Begin Again.wav,True
3,Can't Help Falling In Love,Kina Grannis,Can't Help Falling In Love.wav,../data/raw/youtube_audio/Can't Help Falling I...,True
4,Hold On,Chord Overstreet,Hold On.wav,../data/raw/youtube_audio/Hold On.wav,True
5,Days I Will Remember,Tyrone Wells,Days I Will Remember.wav,../data/raw/youtube_audio/Days I Will Remember...,True
6,Say Something,A Great Big World;Christina Aguilera,Say Something.wav,../data/raw/youtube_audio/Say Something.wav,True
7,I'm Yours,Jason Mraz,I'm Yours.wav,../data/raw/youtube_audio/I'm Yours.wav,True
8,Lucky,Jason Mraz;Colbie Caillat,Lucky.wav,../data/raw/youtube_audio/Lucky.wav,True
9,Hunger,Ross Copperman,Hunger.wav,../data/raw/youtube_audio/Hunger.wav,True


In [17]:
valid_df = df[df["file_exists"]].copy()

print("Total rows in baseline:", len(df))
print("Rows with valid audio files:", len(valid_df))

valid_df[["track_name", "artists", "audio_filename", "audio_path"]].head(10)

Total rows in baseline: 114000
Rows with valid audio files: 10


,track_name,artists,audio_filename,audio_path
0,Comedy,Gen Hoshino,Comedy.wav,../data/raw/youtube_audio/Comedy.wav
1,Ghost - Acoustic,Ben Woodward,Ghost - Acoustic.wav,../data/raw/youtube_audio/Ghost - Acoustic.wav
2,To Begin Again,Ingrid Michaelson;ZAYN,To Begin Again.wav,../data/raw/youtube_audio/To Begin Again.wav
3,Can't Help Falling In Love,Kina Grannis,Can't Help Falling In Love.wav,../data/raw/youtube_audio/Can't Help Falling I...
4,Hold On,Chord Overstreet,Hold On.wav,../data/raw/youtube_audio/Hold On.wav
5,Days I Will Remember,Tyrone Wells,Days I Will Remember.wav,../data/raw/youtube_audio/Days I Will Remember...
6,Say Something,A Great Big World;Christina Aguilera,Say Something.wav,../data/raw/youtube_audio/Say Something.wav
7,I'm Yours,Jason Mraz,I'm Yours.wav,../data/raw/youtube_audio/I'm Yours.wav
8,Lucky,Jason Mraz;Colbie Caillat,Lucky.wav,../data/raw/youtube_audio/Lucky.wav
9,Hunger,Ross Copperman,Hunger.wav,../data/raw/youtube_audio/Hunger.wav


In [18]:
missing_df = df[~df["file_exists"]].copy()
missing_df[["track_name", "artists", "audio_filename", "audio_path"]]

,track_name,artists,audio_filename,audio_path
10,Give Me Your Forever,Zack Tabudlo,,
11,I Won't Give Up,Jason Mraz,,
12,Solo,Dan Berk,,
13,Bad Liar,Anna Hamilton,,
14,Hold On - Remix,Chord Overstreet;Deepend,,
...,...,...,...,...
113995,Sleep My Little Boy,Rainy Lullaby,,
113996,Water Into Light,Rainy Lullaby,,
113997,Miss Perfumado,Cesária Evora,,
113998,Friends,Michael W. Smith,,


In [22]:
def extract_audio_features(file_path):
    y, sr = librosa.load(file_path, sr=22050, mono=True)

    duration_sec = librosa.get_duration(y=y, sr=sr)

    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    tempo = np.asarray(tempo).squeeze()
    tempo = float(tempo) if np.ndim(tempo) == 0 else float(tempo[0])

    rms = librosa.feature.rms(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]

    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)

    pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
    pitch_values = pitches[magnitudes > np.median(magnitudes)]
    pitch_values = pitch_values[pitch_values > 0]

    features = {
        "sample_rate": sr,
        "duration_sec": float(duration_sec),
        "tempo_bpm": float(tempo),
        "rms_mean": float(np.mean(rms)),
        "rms_std": float(np.std(rms)),
        "zcr_mean": float(np.mean(zcr)),
        "zcr_std": float(np.std(zcr)),
        "spectral_centroid_mean": float(np.mean(spectral_centroid)),
        "spectral_centroid_std": float(np.std(spectral_centroid)),
        "spectral_bandwidth_mean": float(np.mean(spectral_bandwidth)),
        "spectral_bandwidth_std": float(np.std(spectral_bandwidth)),
        "spectral_rolloff_mean": float(np.mean(spectral_rolloff)),
        "spectral_rolloff_std": float(np.std(spectral_rolloff)),
        "spectral_contrast_mean": float(np.mean(spectral_contrast)),
        "spectral_contrast_std": float(np.std(spectral_contrast)),
        "pitch_mean": float(np.mean(pitch_values)) if len(pitch_values) > 0 else np.nan,
        "pitch_std": float(np.std(pitch_values)) if len(pitch_values) > 0 else np.nan,
        "pitch_min": float(np.min(pitch_values)) if len(pitch_values) > 0 else np.nan,
        "pitch_max": float(np.max(pitch_values)) if len(pitch_values) > 0 else np.nan,
    }

    for i in range(13):
        features[f"mfcc_{i+1}_mean"] = float(np.mean(mfcc[i]))
        features[f"mfcc_{i+1}_std"] = float(np.std(mfcc[i]))

    for i in range(12):
        features[f"chroma_{i+1}_mean"] = float(np.mean(chroma[i]))
        features[f"chroma_{i+1}_std"] = float(np.std(chroma[i]))

    return features

In [23]:
test_path = valid_df.iloc[0]["audio_path"]
print("Testing:", test_path)

test_features = extract_audio_features(test_path)
test_features

Testing: ../data/raw/youtube_audio/Comedy.wav


{'sample_rate': 22050,
 'duration_sec': 275.34222222222223,
 'tempo_bpm': 89.10290948275862,
 'rms_mean': 0.19171015918254852,
 'rms_std': 0.1307002753019333,
 'zcr_mean': 0.0755513020174551,
 'zcr_std': 0.05753105195972309,
 'spectral_centroid_mean': 2094.675053693486,
 'spectral_centroid_std': 1007.6998383168112,
 'spectral_bandwidth_mean': 2413.529086677553,
 'spectral_bandwidth_std': 760.5382018404792,
 'spectral_rolloff_mean': 4719.810662244261,
 'spectral_rolloff_std': 2125.231646142838,
 'spectral_contrast_mean': 22.987475928737652,
 'spectral_contrast_std': 10.814196693929905,
 'pitch_mean': 984.9515380859375,
 'pitch_std': 932.0209350585938,
 'pitch_min': 145.3815155029297,
 'pitch_max': 3999.709716796875,
 'mfcc_1_mean': -133.7902374267578,
 'mfcc_1_std': 117.74716186523438,
 'mfcc_2_mean': 81.4211654663086,
 'mfcc_2_std': 40.823116302490234,
 'mfcc_3_mean': 21.531354904174805,
 'mfcc_3_std': 23.55425262451172,
 'mfcc_4_mean': 27.66118049621582,
 'mfcc_4_std': 20.996454238891

In [24]:
for idx, row in tqdm(valid_df.iterrows(), total=len(valid_df)):
    try:
        features = extract_audio_features(row["audio_path"])

        for key, value in features.items():
            valid_df.at[idx, key] = value

        valid_df.at[idx, "audio_loaded"] = True
        valid_df.at[idx, "feature_extraction_success"] = True
        valid_df.at[idx, "error_message"] = ""

    except Exception as e:
        valid_df.at[idx, "audio_loaded"] = False
        valid_df.at[idx, "feature_extraction_success"] = False
        valid_df.at[idx, "error_message"] = str(e)

valid_df.head()

  0%|          | 0/10 [00:00<?, ?it/s]

C:\Users\Ammar\AppData\Local\Temp\ipykernel_37484\459714294.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  valid_df.at[idx, "error_message"] = ""


,track_id,track_name,artists,album_name,track_genre,popularity,audio_filename,audio_path,audio_source,audio_loaded,...,chroma_9_std,chroma_10_mean,chroma_10_std,chroma_11_mean,chroma_11_std,chroma_12_mean,chroma_12_std,feature_extraction_success,error_message,file_exists
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino,Comedy,acoustic,73,Comedy.wav,../data/raw/youtube_audio/Comedy.wav,youtube_audio_local,True,...,0.349195,0.479884,0.346018,0.393583,0.299614,0.392972,0.320821,True,,True
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward,Ghost (Acoustic),acoustic,55,Ghost - Acoustic.wav,../data/raw/youtube_audio/Ghost - Acoustic.wav,youtube_audio_local,True,...,0.360636,0.220824,0.209661,0.233700,0.321811,0.157210,0.182448,True,,True
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN,To Begin Again,acoustic,57,To Begin Again.wav,../data/raw/youtube_audio/To Begin Again.wav,youtube_audio_local,True,...,0.293577,0.391381,0.344988,0.322545,0.266624,0.414435,0.304537,True,,True
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,acoustic,71,Can't Help Falling In Love.wav,../data/raw/youtube_audio/Can't Help Falling I...,youtube_audio_local,True,...,0.294844,0.261250,0.335349,0.231623,0.279963,0.309319,0.305553,True,,True
4,5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,Hold On,acoustic,82,Hold On.wav,../data/raw/youtube_audio/Hold On.wav,youtube_audio_local,True,...,0.236605,0.401323,0.330690,0.292185,0.224769,0.358871,0.359677,True,,True


In [26]:
OUTPUT_PATH = "../data/raw/youtube_audio_features_sample.csv"
valid_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to: {OUTPUT_PATH}")

Saved to: ../data/raw/youtube_audio_features_sample.csv


In [27]:
valid_df[
    [
        "track_name",
        "artists",
        "audio_filename",
        "feature_extraction_success",
        "tempo_bpm",
        "pitch_mean",
        "spectral_centroid_mean"
    ]
].head(10)

,track_name,artists,audio_filename,feature_extraction_success,tempo_bpm,pitch_mean,spectral_centroid_mean
0,Comedy,Gen Hoshino,Comedy.wav,True,89.102909,984.951538,2094.675054
1,Ghost - Acoustic,Ben Woodward,Ghost - Acoustic.wav,True,151.999081,730.289246,1512.051492
2,To Begin Again,Ingrid Michaelson;ZAYN,To Begin Again.wav,True,143.554688,888.141235,1886.097292
3,Can't Help Falling In Love,Kina Grannis,Can't Help Falling In Love.wav,True,99.384014,933.427856,1591.025964
4,Hold On,Chord Overstreet,Hold On.wav,True,123.046875,830.291016,1447.016518
5,Days I Will Remember,Tyrone Wells,Days I Will Remember.wav,True,99.384014,941.301025,2440.112174
6,Say Something,A Great Big World;Christina Aguilera,Say Something.wav,True,143.554688,996.367981,1399.990239
7,I'm Yours,Jason Mraz,I'm Yours.wav,True,151.999081,988.373047,1910.333014
8,Lucky,Jason Mraz;Colbie Caillat,Lucky.wav,True,129.199219,987.598389,1968.176677
9,Hunger,Ross Copperman,Hunger.wav,True,161.499023,1075.820557,1428.905237


In [28]:
valid_df[["tempo_bpm", "pitch_mean", "spectral_centroid_mean"]].describe()

,tempo_bpm,pitch_mean,spectral_centroid_mean
count,10.000000,10.000000,10.000000
mean,129.272359,935.656189,1767.838366
std,25.643948,98.212335,346.891554
min,89.102909,730.289246,1399.990239
25%,105.299730,899.462891,1463.275262
50%,136.376953,963.126282,1738.561628
75%,149.887983,988.179382,1953.715761
max,161.499023,1075.820557,2440.112174
